# Person Re-Identification: Experimental Demonstration and Analysis

## 1. Mục tiêu và phạm vi notebook

Notebook này trình bày quy trình thực nghiệm và phân tích định lượng - định tính cho bài toán Person Re-Identification trong bối cảnh trình bày đồ án trên Google Colab. Mục tiêu là hệ thống hóa toàn bộ quy trình đánh giá mô hình, từ kiểm tra dữ liệu, checkpoint, cấu hình, đến trực quan hóa kết quả truy hồi và các tín hiệu nội bộ của mô hình.

Cụ thể, tài liệu này phục vụ các mục tiêu sau:

1. Minh họa bản chất bài toán Person ReID trong thiết lập query-gallery.
2. Đánh giá mô hình baseline TransReID và ba nhánh cải tiến được đề xuất.
3. Phân tích định lượng thông qua các chỉ số mAP, Rank-1, Rank-5 và Rank-10.
4. Phân tích định tính thông qua Top-K retrieval, reliability heatmap và semantic alignment.
5. Đảm bảo khả năng tái lập thực nghiệm thông qua kiểm tra checkpoint, cấu hình và các lệnh chạy tương ứng.

Notebook được tổ chức theo hướng học thuật, ưu tiên tính nhất quán của luận điểm, khả năng kiểm chứng và sự ổn định khi trình chiếu trong video demo. Phần huấn luyện được giữ ở dạng phụ lục tham khảo, trong khi trọng tâm trình bày đặt vào đánh giá từ checkpoint và diễn giải kết quả.

## 2. Bài toán Person Re-Identification

Person Re-Identification được phát biểu như một bài toán truy hồi hình ảnh có giám sát yếu trong môi trường đa camera. Với một ảnh **query** (ảnh người cần tìm), hệ thống phải truy hồi trong tập **gallery** (ảnh ứng viên thu từ các camera khác nhau) và sắp xếp các ứng viên theo mức độ tương đồng với query.

Về mặt biểu diễn, mỗi ảnh được ánh xạ thành một **embedding** trong không gian đặc trưng. Từ đó, một hàm khoảng cách hoặc độ tương đồng (distance/similarity) được sử dụng để tạo **ranking** các ảnh gallery đối với từng query. Do đó, ReID khác biệt căn bản so với classification: đầu ra không phải một nhãn lớp duy nhất, mà là một danh sách ứng viên có thứ hạng.

Bài toán này đặc biệt thách thức do đồng thời chịu tác động của nhiều yếu tố: sai khác miền giữa camera (cross-camera shift), biến thiên tư thế (pose variation), thay đổi điều kiện chiếu sáng (illumination), che khuất cục bộ (occlusion), nhiễu nền (background clutter), và hiện tượng nhiều người có ngoại hình tương tự nhau (visual ambiguity). Vì vậy, việc phân tích đồng thời kết quả định lượng và định tính là cần thiết để hiểu hành vi mô hình một cách đầy đủ.

## 3. Thiết lập thực nghiệm và tham số điều khiển

Các biến `RUN_*` quyết định notebook sẽ thực thi lệnh tính toán hay chỉ hiển thị cấu hình và artifact đã có sẵn. Trong notebook demo chính thức, các artifact định lượng và định tính có thể được sinh trước để đảm bảo quá trình quay video ổn định. Khi cần tái lập toàn bộ quy trình, có thể bật lại các cờ `RUN_EVAL` và `RUN_VISUALIZE`.

- `RUN_EVAL = True`: thực thi `test.py` để đánh giá các checkpoint; chế độ này có thể tốn thời gian.
- `RUN_VISUALIZE = True`: sinh lại các artifact hình ảnh phục vụ phân tích định lượng và định tính; chế độ này cũng có thể tốn thời gian.
- `RUN_TRAIN = True`: thực thi lệnh huấn luyện; không khuyến nghị bật khi quay video vì thời gian chạy rất dài.
- Khi quay video, có thể giữ `RUN_EVAL = False` và `RUN_VISUALIZE = False` nếu các artifact đã được sinh trước đó.
- Khi cần tái lập đầy đủ kết quả, nên bật `RUN_EVAL = True` và `RUN_VISUALIZE = True`.

## Kiểm tra môi trường thực thi

Cell này xác nhận notebook đang chạy trên Google Colab, hiển thị phiên bản Python, trạng thái GPU, thư mục làm việc hiện tại và các đường dẫn cốt lõi trước khi chuyển sang preflight check và evaluation.

In [ ]:
# Các đường dẫn dưới đây được cấu hình cho Google Colab.
# Nếu cấu trúc thư mục khác, chỉ cần chỉnh DRIVE_ROOT, PROJECT_ROOT và HF_DATASET_*.

from pathlib import Path
import json
import os
import shlex
import shutil
import stat
import subprocess
import sys
import urllib.request
import zipfile
from collections import defaultdict
from datetime import datetime

from IPython.display import display, Markdown, Image as IPyImage

USE_GOOGLE_DRIVE = True
USE_HF_DOWNLOAD = True
# Hugging Face dataset khuyến nghị để bám sát Market-1501 gốc.
HF_DATASET_REPO_ID = "tuandunghcmut/MyPublicStorage"
HF_DATASET_FILENAME = "Market-1501-v15.09.15.zip"


def is_colab():
    """Xác định notebook hiện có đang chạy trong Google Colab hay không.

    Hàm này được dùng để quyết định có mount Google Drive và in các chẩn đoán
    đặc thù của Colab hay không. Nếu notebook được mở trong môi trường khác,
    hàm vẫn trả về False thay vì phát sinh lỗi.
    """
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def _has_market1501_splits(path):
    """Kiểm tra nhanh một thư mục có đủ cấu trúc split Market1501 hay không."""
    return (
        path.exists()
        and (path / "bounding_box_train").exists()
        and (path / "query").exists()
        and (path / "bounding_box_test").exists()
    )


def _find_market1501_dir(search_root):
    """Tìm thư mục dataset Market1501 trong một thư mục cha sau khi giải nén."""
    fixed_candidates = [
        search_root / "market1501",
        search_root / "Market-1501-v15.09.15",
        search_root / "Market1501",
    ]
    for candidate in fixed_candidates:
        if _has_market1501_splits(candidate):
            return candidate

    if not search_root.exists():
        return None

    for child in search_root.iterdir():
        if child.is_dir() and "market" in child.name.lower() and "1501" in child.name:
            if _has_market1501_splits(child):
                return child
    return None


def _download_file(url, destination):
    """Tải file từ URL về destination một cách an toàn bằng file tạm."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_path = destination.with_suffix(destination.suffix + ".part")

    with urllib.request.urlopen(url) as response, open(temp_path, "wb") as output_file:
        shutil.copyfileobj(response, output_file)

    temp_path.replace(destination)


def ensure_market1501_dataset(dataset_root, hf_repo_id, hf_filename, auto_download=True):
    """Đảm bảo dataset Market1501 tồn tại tại /content bằng cách tải từ Hugging Face nếu cần.

    Parameters
    ----------
    dataset_root : Path
        Thư mục dataset chuẩn hóa mà notebook sẽ sử dụng.
    hf_repo_id : str
        Repo id trên Hugging Face, ví dụ `aveocr/Market-1501-v15.09.15.zip`.
    hf_filename : str
        Tên file zip trong repo dataset.
    auto_download : bool
        Nếu True, tự động tải và giải nén khi dataset chưa tồn tại.

    Returns
    -------
    Path
        Đường dẫn dataset đã sẵn sàng cho các cell phía sau.

    Raises
    ------
    RuntimeError
        Nếu thiếu repo id/filename hợp lệ hoặc tải/giải nén thất bại.
    """
    dataset_root = Path(dataset_root)
    if _has_market1501_splits(dataset_root):
        print(f"[OK] Dataset found: {dataset_root}")
        return dataset_root

    if not auto_download:
        print(f"[WARNING] Dataset not found and auto download is disabled: {dataset_root}")
        return dataset_root

    if not hf_repo_id or "/" not in hf_repo_id:
        raise RuntimeError(
            "HF_DATASET_REPO_ID không hợp lệ. Hãy đặt theo dạng owner/dataset-repo-name."
        )
    if not hf_filename.lower().endswith(".zip"):
        raise RuntimeError("HF_DATASET_FILENAME phải là một file .zip hợp lệ.")

    download_root = dataset_root.parent
    download_root.mkdir(parents=True, exist_ok=True)

    archive_path = download_root / hf_filename
    extract_root = download_root / "_hf_market1501_extract"

    hf_url = f"https://huggingface.co/datasets/{hf_repo_id}/resolve/main/{hf_filename}?download=1"
    print(f"[INFO] Downloading Market1501 from Hugging Face: {hf_repo_id}")
    print(f"[INFO] URL: {hf_url}")

    if not archive_path.exists():
        _download_file(hf_url, archive_path)
    else:
        print(f"[INFO] Reusing existing archive: {archive_path}")

    if extract_root.exists():
        shutil.rmtree(extract_root, ignore_errors=True)
    extract_root.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Extracting archive: {archive_path}")
    with zipfile.ZipFile(archive_path, "r") as zip_ref:
        zip_ref.extractall(extract_root)

    discovered = _find_market1501_dir(extract_root)
    if discovered is None:
        discovered = _find_market1501_dir(download_root)
    if discovered is None:
        raise RuntimeError(
            f"Đã tải nhưng không tìm thấy cấu trúc Market1501 trong: {extract_root}"
        )

    if discovered != dataset_root:
        if dataset_root.exists() and not _has_market1501_splits(dataset_root):
            shutil.rmtree(dataset_root, ignore_errors=True)
        if not dataset_root.exists():
            try:
                discovered.rename(dataset_root)
            except OSError:
                shutil.copytree(discovered, dataset_root, dirs_exist_ok=True)

    if not _has_market1501_splits(dataset_root):
        raise RuntimeError(f"Dataset chưa hợp lệ sau khi chuẩn hóa: {dataset_root}")

    shutil.rmtree(extract_root, ignore_errors=True)
    print(f"[OK] Dataset ready at: {dataset_root}")
    return dataset_root


if USE_GOOGLE_DRIVE and is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
elif USE_GOOGLE_DRIVE:
    print("[WARNING] Google Drive mount skipped because the runtime is not Colab.")

# Đường dẫn runtime và Drive trên Google Colab.
COLAB_ROOT = Path("/content")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_ROOT / "MyDrive"

# Đường dẫn dự án, mã nguồn và artifact.
PROJECT_ROOT = DRIVE_ROOT / "K23-TGMT"
SOURCE_ROOT = PROJECT_ROOT / "1.3 Source code"
DEMO_ROOT = PROJECT_ROOT / "1.4 Demo"

# Quan trọng: checkpoint/config/visualization được đọc từ transreid_artifacts.
ARTIFACT_ROOT = SOURCE_ROOT / "transreid_artifacts"

LOCAL_SRC = SOURCE_ROOT / "local-reliability"
SEMANTIC_SRC = SOURCE_ROOT / "semantic"
TOOLS_DIR = ARTIFACT_ROOT / "tools"

# Dataset không còn đọc từ Drive. Notebook sẽ dùng dữ liệu local trong /content để tăng tốc I/O.
DATASET_ROOT = ensure_market1501_dataset(
    dataset_root=Path("/content/data/market1501"),
    hf_repo_id=HF_DATASET_REPO_ID,
    hf_filename=HF_DATASET_FILENAME,
    auto_download=USE_HF_DOWNLOAD,
)
DATA_PARENT = DATASET_ROOT.parent

# Output chạy demo được ghi tách riêng để không ghi đè artifact gốc.
DEMO_OUTPUT = DEMO_ROOT / "generated_outputs"
DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)

# Các cờ điều khiển thực nghiệm. Giữ giá trị False khi trình chiếu để tránh chạy tác vụ nặng.
RUN_INSTALL = False
RUN_TRAIN = False
FAST_TRAIN_DEMO = True
RUN_EVAL = False
RUN_VISUALIZE = False
STRICT_PREFLIGHT = True

DEVICE_ID = "0"
PYTHON = sys.executable


def print_runtime_diagnostics():
    """In chẩn đoán môi trường thực thi để phục vụ demo và kiểm tra nhanh.

    Nội dung bao gồm trạng thái Colab, phiên bản Python, thư mục làm việc hiện tại,
    các đường dẫn quan trọng và thông tin GPU nếu nvidia-smi khả dụng. Hàm này
    được thiết kế theo hướng an toàn: thiếu GPU hoặc thiếu lệnh hệ thống không làm
    notebook bị dừng.
    """
    print("Running on Colab:", is_colab())
    print("Python:", sys.version)
    print("Current working directory:", os.getcwd())
    print("Drive root exists:", DRIVE_ROOT.exists(), DRIVE_ROOT)
    print("Project root:", PROJECT_ROOT)
    print("Dataset root:", DATASET_ROOT)
    print("Artifact root:", ARTIFACT_ROOT)
    try:
        result = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True)
        if result.stdout.strip():
            print(result.stdout)
        else:
            print("nvidia-smi is unavailable or no NVIDIA GPU is exposed.")
    except FileNotFoundError:
        print("nvidia-smi is unavailable or no NVIDIA GPU is exposed.")


print_runtime_diagnostics()

Trong notebook demo chính thức, các artifact định lượng và định tính có thể được sinh trước để đảm bảo quá trình quay video ổn định. Khi cần tái lập toàn bộ quy trình, có thể bật lại các cờ `RUN_EVAL` và `RUN_VISUALIZE`.

## 4. Baseline và ba nhánh phương pháp đề xuất

Thực nghiệm sử dụng **TransReID** làm mô hình nền (baseline) để làm mốc so sánh nhất quán giữa các biến thể. Cấu trúc so sánh được giữ cố định nhằm bảo đảm mọi thay đổi về kết quả có thể quy về khác biệt phương pháp thay vì khác biệt ở giao thức đánh giá.

### Baseline TransReID
- Là mô hình nền dùng Transformer để học embedding cho Person Re-Identification.
- Ảnh đầu vào được chia thành các patch token trước khi được mã hóa bởi backbone Transformer.
- Mục tiêu học là đưa các mẫu cùng identity gần nhau trong không gian embedding và đẩy các mẫu khác identity ra xa.
- Đây là mốc tham chiếu cần thiết để đánh giá mức cải thiện của các nhánh đề xuất.

### Local Reliability
- Xuất phát từ nhận định rằng không phải mọi patch đều mang thông tin định danh như nhau.
- Các vùng như áo, quần, balo, giày và dáng người thường hữu ích hơn so với nền hoặc các vùng bị che khuất.
- Nhánh này điều chỉnh mức đóng góp của patch/local token thông qua tín hiệu reliability.
- Mục tiêu là giảm tác động của các vùng không ổn định và tăng độ bền của embedding trước nhiễu cục bộ.

### Semantic Alignment
- Nhánh này khai thác semantic/body-part mask để đưa thêm cấu trúc cơ thể vào quá trình học embedding.
- Tín hiệu semantic giúp mô hình phân biệt rõ hơn vùng người, vùng nền và các vùng cơ thể quan trọng.
- Cách tiếp cận này có thể hỗ trợ ổn định khi pose, camera hoặc background thay đổi.
- Về mặt diễn giải, semantic alignment đóng vai trò là một tín hiệu phụ trợ có cấu trúc chứ không phải bằng chứng rằng mô hình hiểu hoàn toàn hình thái cơ thể.

### Semantic + Reliability
- Cấu hình này kết hợp semantic supervision và reliability-aware weighting.
- Mục tiêu là vừa tăng tính có cấu trúc của patch token, vừa giảm ảnh hưởng của vùng kém tin cậy.
- Đây là cấu hình tổng hợp để đánh giá tác động kết hợp của hai hướng cải tiến.
- Nhánh này cung cấp góc nhìn đầy đủ hơn về khả năng bổ trợ giữa tín hiệu ngữ nghĩa và trọng số tin cậy cục bộ.

Cách thiết kế này cho phép phân tích đóng góp của từng thành phần một cách rõ ràng thông qua so sánh định lượng và định tính.

In [ ]:
# Cell này khai báo baseline và các nhánh đề xuất dưới dạng dictionary thống nhất.
# Cấu trúc BRANCHES giúp tránh hard-code lặp lại ở các bước evaluate/train/visualize.
BRANCHES = {
    "baseline": {
        "title": "Baseline TransReID",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_fair.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_baseline" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_baseline",
        "note": "Mô hình nền TransReID.",
    },
    "local": {
        "title": "Local Reliability",
        "source": LOCAL_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_local_reliability.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_local_reliability" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_local_reliability",
        "note": "Ước lượng reliability score cho patch/local token.",
    },
    "semantic": {
        "title": "Semantic Alignment",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_sem_align.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_sem_align" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_semantic_alignment",
        "note": "Dùng semantic mask để giám sát patch token.",
    },
    "combine": {
        "title": "Semantic + Reliability",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_sem_align_reliability.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_sem_align_reliability" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_semantic_reliability",
        "note": "Kết hợp semantic supervision và reliability-aware local fusion.",
    },
}

rows = []
for key, branch in BRANCHES.items():
    ckpt = branch["checkpoint"]
    cfg = branch["config"]
    rows.append({
        "key": key,
        "model": branch["title"],
        "source": str(branch["source"]),
        "config_exists": cfg.exists(),
        "checkpoint_exists": ckpt.exists(),
        "checkpoint_size_MB": round(ckpt.stat().st_size / 1024**2, 2) if ckpt.exists() else None,
        "note": branch["note"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    # Fallback in text mode để notebook vẫn hoạt động khi thiếu pandas.
    for row in rows:
        print(row)

## 5. Kiểm tra điều kiện thực nghiệm

Mục này thực hiện kiểm tra tính sẵn sàng của toàn bộ pipeline trước khi đánh giá:

- Đường dẫn dự án, mã nguồn, artifact và thư mục công cụ.
- Cấu trúc dữ liệu ReID và số lượng ảnh theo từng split.
- Cấu hình và checkpoint của bốn nhánh mô hình.
- Các script trực quan hóa quan trọng dùng cho phân tích định tính.

Ký hiệu trạng thái:

- `[OK]`: thành phần sẵn sàng cho thực nghiệm.
- `[WARNING]`: thành phần phụ còn thiếu, có thể ảnh hưởng một phần phân tích.
- `[MISSING]`: thiếu thành phần thiết yếu, cần khắc phục trước khi chạy các bước chính.

In [ ]:
# Cell này thực hiện kiểm tra điều kiện thực nghiệm trước khi chạy evaluation hoặc visualization.
# Thiết kế theo hướng fail-fast cho thành phần bắt buộc, nhưng vẫn cho phép notebook tiếp tục khi thiếu artifact phụ.
def count_images(split_name):
    """Đếm số ảnh .jpg trong một split của dataset.

    Parameters
    ----------
    split_name : str
        Tên thư mục split, ví dụ `query` hoặc `bounding_box_test`.

    Returns
    -------
    int
        Số lượng ảnh .jpg trong split. Hàm trả về 0 nếu split không tồn tại.

    Why it exists
    ------------
    Notebook cần thống kê nhanh quy mô dataset để phục vụ phần trình bày và để
    preflight check có thể báo cáo tình trạng dữ liệu mà không làm notebook bị dừng.
    """
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return 0
    return len(list(split_dir.glob("*.jpg")))


def preflight_check(verbose=True, strict=True):
    """Kiểm tra môi trường Colab, dataset, checkpoint, cấu hình và artifact.

    Parameters
    ----------
    verbose : bool
        Nếu True, in báo cáo chi tiết ra output cell để tiện thuyết trình.
    strict : bool
        Nếu True, raise lỗi khi thiếu thành phần bắt buộc cho evaluation/demo.
        Nếu False, chỉ ghi cảnh báo và trả về danh sách mục thiếu.

    Returns
    -------
    dict
        Từ điển chứa `logs`, `hard_missing`, `warnings` và các thống kê phụ.

    Behavior
    --------
    - Kiểm tra Google Drive đã mount và các đường dẫn cốt lõi đã tồn tại.
    - Kiểm tra ba split dữ liệu chính cùng số lượng ảnh.
    - Kiểm tra config và checkpoint của 4 nhánh mô hình.
    - Kiểm tra các artifact định lượng/định tính nếu notebook dùng lại ảnh hoặc CSV có sẵn.
    - Không raise lỗi đối với artifact phụ; chỉ raise khi thiếu thành phần bắt buộc cho evaluation.
    """
    logs = []
    hard_missing = []
    warnings = []
    split_stats = {}

    def push(status, label, path=None, required=False, detail=""):
        msg = f"[{status}] {label}"
        if path is not None:
            msg += f" -> {path}"
        if detail:
            msg += f" ({detail})"
        logs.append(msg)
        if status == "WARNING":
            warnings.append(msg)
        if status == "MISSING" and required:
            hard_missing.append(msg)

    drive_ready = DRIVE_ROOT.exists()
    push("OK" if drive_ready else "MISSING", "Google Drive mount", DRIVE_ROOT, required=True)

    core_paths = [
        ("PROJECT_ROOT", PROJECT_ROOT, True),
        ("SOURCE_ROOT", SOURCE_ROOT, True),
        ("ARTIFACT_ROOT", ARTIFACT_ROOT, True),
        ("DEMO_OUTPUT", DEMO_OUTPUT, True),
        ("LOCAL_SRC", LOCAL_SRC, True),
        ("SEMANTIC_SRC", SEMANTIC_SRC, True),
        ("TOOLS_DIR", TOOLS_DIR, False),
    ]
    for name, path, required in core_paths:
        push("OK" if path.exists() else "MISSING", name, path, required=required)

    push("OK" if DATASET_ROOT.exists() else "MISSING", "DATASET_ROOT", DATASET_ROOT, required=True)
    for split in ["bounding_box_train", "query", "bounding_box_test"]:
        split_dir = DATASET_ROOT / split
        if split_dir.exists():
            image_count = count_images(split)
            split_stats[split] = image_count
            push("OK", f"Split {split}", split_dir, detail=f"{image_count} images")
        else:
            split_stats[split] = 0
            required = split in {"query", "bounding_box_test"}
            push("MISSING", f"Split {split}", split_dir, required=required)

    for key, branch in BRANCHES.items():
        cfg = branch["config"]
        ckpt = branch["checkpoint"]
        push("OK" if cfg.exists() else "MISSING", f"{key}.config", cfg, required=True)
        if ckpt.exists():
            ckpt_mb = round(ckpt.stat().st_size / 1024**2, 2)
            push("OK", f"{key}.checkpoint", ckpt, required=True, detail=f"{ckpt_mb} MB")
        else:
            push("MISSING", f"{key}.checkpoint", ckpt, required=True)

    optional_artifacts = [
        ("artifact.metrics_summary", ARTIFACT_ROOT / "metrics_summary.csv"),
        ("artifact.latency_summary", ARTIFACT_ROOT / "latency_summary.csv"),
        ("artifact.quantitative_dir", ARTIFACT_ROOT / "visualizations" / "quantitative"),
        ("artifact.qualitative_dir", ARTIFACT_ROOT / "visualizations" / "qualitative"),
        ("artifact.local_reliability_dir", ARTIFACT_ROOT / "visualizations" / "local_reliability"),
        ("artifact.semantic_dir", ARTIFACT_ROOT / "visualizations" / "semantic"),
    ]
    for label, path in optional_artifacts:
        if path.exists():
            push("OK", label, path)
        else:
            push("WARNING", label, path)

    if verbose:
        print("Running on Colab:", is_colab())
        print("Python:", sys.version.split()[0])
        print("Current working directory:", os.getcwd())
        print("Drive root:", DRIVE_ROOT)
        print("Project root:", PROJECT_ROOT)
        print("Dataset root:", DATASET_ROOT)
        print("Artifact root:", ARTIFACT_ROOT)
        print("Split statistics:", split_stats)
        try:
            import torch
            print("Torch:", torch.__version__)
            print("CUDA available:", torch.cuda.is_available())
            if torch.cuda.is_available():
                print("GPU:", torch.cuda.get_device_name(0))
        except Exception as exc:
            print("[WARNING] Torch check failed:", exc)

        print("\n--- Preflight Summary ---")
        for line in logs:
            print(line)

    if hard_missing and strict:
        raise RuntimeError(
            "Thiếu thành phần bắt buộc cho demo/evaluation. Hãy xử lý các mục [MISSING] sau:\n"
            + "\n".join(hard_missing)
        )

    return {
        "logs": logs,
        "hard_missing": hard_missing,
        "warnings": warnings,
        "split_stats": split_stats,
        "drive_ready": drive_ready,
    }


_ = preflight_check(verbose=True, strict=STRICT_PREFLIGHT)

### Hàm tiện ích nội bộ

Các hàm dưới đây hỗ trợ thực thi lệnh, chuẩn hóa hiển thị artifact và giảm trùng lặp mã nguồn.
Việc tách thành hàm riêng giúp notebook rõ cấu trúc và thuận lợi cho tái lập thực nghiệm.

In [ ]:
# Cell tiện ích dùng chung cho toàn notebook: chuẩn hóa command, chạy subprocess, hiển thị ảnh và CSV.
# Tách các thao tác này ra thành helper giúp notebook gọn hơn và dễ tái sử dụng khi quay video demo.
def quote_cmd(cmd):
    """Chuẩn hóa danh sách tham số lệnh thành chuỗi dễ đọc.

    Parameters
    ----------
    cmd : sequence
        Danh sách thành phần lệnh, có thể chứa `str`, `Path` hoặc số.

    Returns
    -------
    str
        Chuỗi command-line đã được quote an toàn để hiển thị.

    Why it exists
    ------------
    Notebook demo cần in lệnh theo cách ổn định để người xem có thể đọc và tái lập
    pipeline mà không phụ thuộc vào cách Python render từng đối tượng.
    """
    return " ".join(shlex.quote(str(x)) for x in cmd)


def run_command(cmd, cwd, run=False, env=None):
    """Thực thi một lệnh shell bằng subprocess với chế độ bật/tắt.

    Parameters
    ----------
    cmd : sequence
        Danh sách tham số lệnh cho subprocess.
    cwd : Path-like
        Thư mục làm việc khi chạy lệnh.
    run : bool
        Nếu False, chỉ in command để tham khảo; nếu True thì thực thi.
    env : dict or None
        Các biến môi trường bổ sung sẽ merge với `os.environ`.

    Returns
    -------
    subprocess.CompletedProcess or None
        Kết quả lệnh khi `run=True`; `None` nếu notebook chỉ in command.

    Error handling
    --------------
    - Nếu lệnh thất bại, hàm raise `RuntimeError` để notebook dừng ở đúng vị trí lỗi.
    - Nếu `run=False`, hàm không làm thay đổi trạng thái notebook và chỉ đóng vai trò ghi chú thực nghiệm.
    """
    print("CWD:", cwd)
    print("CMD:", quote_cmd(cmd))
    if not run:
        print("Skip: đặt cờ RUN_* = True để chạy lệnh này.")
        return None
    env_all = os.environ.copy()
    if env:
        env_all.update(env)
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=env_all,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    # Chỉ in phần cuối của output để tránh tràn màn hình khi quay video hoặc xem lại notebook.
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with return code {proc.returncode}")
    return proc


def show_image(path, width=1000):
    """Hiển thị ảnh nếu tồn tại, đồng thời báo rõ khi artifact chưa sẵn sàng.

    Parameters
    ----------
    path : Path-like
        Đường dẫn tới ảnh cần hiển thị.
    width : int
        Chiều rộng hiển thị trong notebook.

    Returns
    -------
    None
        Hàm chỉ tạo side effect hiển thị.

    Why it exists
    ------------
    Notebook dùng nhiều artifact hình ảnh; helper này giúp không crash khi ảnh
    chưa được sinh và vẫn giữ luồng trình bày liền mạch.
    """
    path = Path(path)
    if not path.exists():
        display(Markdown(f"**Chưa có ảnh:** `{path}`"))
        return
    display(Markdown(f"`{path}`"))
    display(IPyImage(filename=str(path), width=width))


def show_csv(path):
    """Đọc và hiển thị tệp CSV theo định dạng bảng.

    Parameters
    ----------
    path : Path-like
        Đường dẫn tệp CSV.

    Returns
    -------
    None
        Nếu CSV không tồn tại hoặc thiếu pandas, hàm sẽ in thông báo thay vì dừng notebook.

    Why it exists
    ------------
    Báo cáo metric trong notebook phụ thuộc vào CSV đã tổng hợp từ evaluation;
    helper này giữ notebook an toàn khi artifact chưa có sẵn.
    """
    path = Path(path)
    if not path.exists():
        print("Missing:", path)
        return
    try:
        import pandas as pd
        display(pd.read_csv(path))
    except Exception:
        print(path.read_text(encoding="utf-8-sig"))

## 6. Khảo sát dữ liệu và minh họa cùng danh tính qua nhiều camera

Mục này trực quan hóa tính chất cốt lõi của ReID:

1. Cùng một danh tính (`pid`) có thể xuất hiện rất khác nhau dưới các camera khác nhau.
2. Nhiều danh tính khác có thể mang ngoại hình tương tự, tạo nhiễu đối với truy hồi.

Notebook sẽ parse `pid`/`camid` từ tên tệp ảnh theo chuẩn thường dùng của Market-1501/Duke, sau đó hiển thị:

- Một nhóm ảnh cùng `pid` (ưu tiên đa camera).
- Một nhóm distractor khác `pid` để minh họa độ mơ hồ thị giác.

In [ ]:
# Cell này xây dựng các hàm hỗ trợ khảo sát dữ liệu và minh họa same-identity cross-camera.
# Mục tiêu là cung cấp bằng chứng trực quan về độ khó của bài toán ReID trước khi vào đánh giá mô hình.
def parse_pid_camid(img_path):
    """Trích xuất pid và camid từ tên tệp ảnh theo chuẩn Market-1501/Duke.

    Args:
        img_path: Đường dẫn ảnh hoặc tên tệp ảnh.

    Returns:
        Tuple (pid, camid). Nếu không parse được camid, trả về 'unknown'.
    """
    stem = Path(img_path).stem
    parts = stem.split("_")
    pid = parts[0] if parts else "unknown"
    camid = "unknown"
    for part in parts[1:]:
        if part.startswith("c") and len(part) > 1 and part[1].isdigit():
            camid = part[1:]
            break
    return pid, camid


def list_split_images(split_name, limit=None):
    """Liệt kê ảnh trong một split dataset theo thứ tự ổn định.

    Args:
        split_name: Tên split (ví dụ: 'query', 'bounding_box_test').
        limit: Giới hạn số ảnh trả về; None nghĩa là lấy toàn bộ.

    Returns:
        Danh sách Path đến tệp .jpg. Trả về list rỗng nếu split không tồn tại.
    """
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return []
    images = sorted(split_dir.glob("*.jpg"))
    if limit is not None:
        return images[:limit]
    return images


def build_pid_index(images):
    """Tạo chỉ mục pid -> danh sách (image_path, camid).

    Args:
        images: Danh sách đường dẫn ảnh.

    Returns:
        defaultdict(list) ánh xạ pid sang các mẫu ảnh tương ứng.

    Notes:
        Bỏ qua pid '-1', '0000' và giá trị không hợp lệ để giảm nhiễu.
    """
    index = defaultdict(list)
    for p in images:
        pid, camid = parse_pid_camid(p)
        if pid in {"-1", "0000", "unknown"}:
            continue
        index[pid].append((p, camid))
    return index


def show_same_id_cross_camera(max_same_id=6, max_distractor=6):
    """Hiển thị cùng danh tính qua nhiều camera và một hàng distractor.

    Args:
        max_same_id: Số ảnh tối đa của cùng pid ở hàng trên.
        max_distractor: Số ảnh tối đa khác pid ở hàng dưới.

    Returns:
        None. Hàm hiển thị trực tiếp bằng matplotlib.

    Error Handling:
        - In cảnh báo nếu thiếu matplotlib/Pillow.
        - In cảnh báo rõ khi thiếu dữ liệu đủ điều kiện.
    """
    try:
        import matplotlib.pyplot as plt
        from PIL import Image
    except Exception as exc:
        print("[WARNING] Missing matplotlib/Pillow for preview:", exc)
        return

    gallery_images = list_split_images("bounding_box_test")
    if not gallery_images:
        print("[MISSING] Không tìm thấy ảnh trong split bounding_box_test.")
        return

    pid_index = build_pid_index(gallery_images)
    candidates = []
    for pid, items in pid_index.items():
        cams = {cam for _, cam in items}
        if len(items) >= 4:
            candidates.append((pid, len(cams), len(items), items))

    if not candidates:
        print("[WARNING] Không đủ identity có >=4 ảnh để minh họa cross-camera.")
        return

    # Ưu tiên identity có nhiều camera nhất, sau đó nhiều ảnh nhất.
    candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
    chosen_pid, num_cams, _, items = candidates[0]

    same_samples = sorted(items, key=lambda x: x[1])[:max_same_id]

    other_pids = [pid for pid in pid_index.keys() if pid != chosen_pid]
    distractors = []
    for pid in other_pids:
        if len(distractors) >= max_distractor:
            break
        distractors.append((pid_index[pid][0][0], parse_pid_camid(pid_index[pid][0][0])[1], pid))

    n_cols = max(len(same_samples), len(distractors), 4)
    fig, axes = plt.subplots(2, n_cols, figsize=(3.2 * n_cols, 8))

    for c in range(n_cols):
        axes[0, c].axis("off")
        axes[1, c].axis("off")

    for c, (img_path, camid) in enumerate(same_samples):
        axes[0, c].imshow(Image.open(img_path).convert("RGB"))
        axes[0, c].set_title(f"same pid={chosen_pid}\ncam={camid}", fontsize=10)

    for c, (img_path, camid, pid) in enumerate(distractors):
        axes[1, c].imshow(Image.open(img_path).convert("RGB"))
        axes[1, c].set_title(f"distractor pid={pid}\ncam={camid}", fontsize=10)

    fig.suptitle(
        f"Same identity across cameras (pid={chosen_pid}, cams={num_cams}) vs distractors",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


show_same_id_cross_camera(max_same_id=6, max_distractor=6)

## 7. Thiết lập query-gallery

Phần này minh họa cấu hình truy hồi chuẩn:

- Một ảnh query đóng vai trò mẫu cần tìm.
- Tập gallery gồm các ứng viên có thể chứa cả true match và distractor.

Kết quả của mô hình sẽ là danh sách gallery được xếp hạng theo mức độ tương đồng embedding với query. Đây là cơ sở để diễn giải các chỉ số Rank-k và phân tích Top-K retrieval ở các mục sau.

In [ ]:
# Cell này minh họa setup query-gallery với một ví dụ cụ thể (1 query + nhiều candidate).
# Việc quan sát mẫu này giúp người đọc hiểu trực tiếp cơ chế ranking trước khi xem Top-K của mô hình.
def show_query_gallery_example(num_gallery=5):
    """Hiển thị một ví dụ query-gallery gồm positive và distractor.

    Args:
        num_gallery: Số lượng ảnh gallery cần hiển thị (bao gồm 1 positive nếu tìm được).

    Returns:
        None. Hàm hiển thị bằng matplotlib.

    Error Handling:
        - Cảnh báo nếu thiếu thư viện hiển thị.
        - Cảnh báo rõ khi không có dữ liệu query/gallery hợp lệ.
    """
    try:
        import matplotlib.pyplot as plt
        from PIL import Image
    except Exception as exc:
        print("[WARNING] Missing matplotlib/Pillow for query-gallery preview:", exc)
        return

    query_images = list_split_images("query")
    gallery_images = list_split_images("bounding_box_test")

    if not query_images or not gallery_images:
        print("[MISSING] Cần có cả split query và bounding_box_test để minh họa query-gallery.")
        return

    gallery_index = build_pid_index(gallery_images)

    selected_query = None
    selected_positive = None
    for q in query_images:
        q_pid, q_cam = parse_pid_camid(q)
        if q_pid in {"-1", "0000", "unknown"}:
            continue
        positives = [item for item in gallery_index.get(q_pid, []) if item[0].name != q.name]
        if positives:
            selected_query = q
            selected_positive = positives[0][0]
            break

    if selected_query is None:
        print("[WARNING] Không tìm thấy query có positive match trong gallery.")
        return

    q_pid, q_cam = parse_pid_camid(selected_query)

    negatives = []
    for g in gallery_images:
        g_pid, g_cam = parse_pid_camid(g)
        if g_pid != q_pid and g_pid not in {"-1", "0000", "unknown"}:
            negatives.append(g)
        if len(negatives) >= max(1, num_gallery - 1):
            break

    gallery_show = [selected_positive] + negatives

    n_cols = 1 + len(gallery_show)
    fig, axes = plt.subplots(1, n_cols, figsize=(4.0 * n_cols, 5.2))

    axes[0].imshow(Image.open(selected_query).convert("RGB"))
    axes[0].set_title(f"QUERY\npid={q_pid} cam={q_cam}", fontsize=11)
    axes[0].axis("off")

    for i, g in enumerate(gallery_show, start=1):
        g_pid, g_cam = parse_pid_camid(g)
        is_match = g_pid == q_pid
        axes[i].imshow(Image.open(g).convert("RGB"))
        axes[i].set_title(
            f"GALLERY-{i}\npid={g_pid} cam={g_cam}\n{'MATCH' if is_match else 'DISTRACTOR'}",
            fontsize=10,
        )
        axes[i].axis("off")

    fig.suptitle("Query-Gallery setup for Person Re-ID", fontsize=14)
    plt.tight_layout()
    plt.show()


show_query_gallery_example(num_gallery=5)

## 8. Đánh giá từ checkpoint đã huấn luyện

Đây là bước đánh giá trung tâm của notebook. Với mỗi nhánh mô hình, `test.py` được gọi cùng cấu hình tương ứng để tính toán các chỉ số truy hồi tiêu chuẩn.

Khi đặt `RUN_EVAL = True`, kết quả đánh giá sẽ được ghi vào thư mục `generated_outputs/eval_*`, làm đầu vào cho các bước tổng hợp bảng và trực quan hóa tiếp theo.

In [ ]:
# Cell này xây dựng lệnh evaluation cho từng nhánh từ checkpoint đã huấn luyện.
# Khi RUN_EVAL=False, notebook chỉ hiển thị command để phục vụ thuyết trình và kiểm tra cấu hình.
def build_eval_command(key):
    """Tạo command đánh giá cho một nhánh mô hình.

    Parameters
    ----------
    key : str
        Tên nhánh trong `BRANCHES` (`baseline`, `local`, `semantic`, `combine`).

    Returns
    -------
    list
        Danh sách tham số lệnh dùng cho subprocess.

    Why it exists
    ------------
    Notebook demo cần cùng một giao thức evaluation cho cả bốn nhánh để so sánh
    công bằng các metric mAP và CMC Rank-k.
    """
    branch = BRANCHES[key]
    cmd = [
        PYTHON, "test.py",
        "--config_file", branch["config"],
        "DATASETS.ROOT_DIR", str(DATA_PARENT),
        "MODEL.DEVICE_ID", DEVICE_ID,
        "TEST.WEIGHT", str(branch["checkpoint"]),
        "TEST.IMS_PER_BATCH", "128",
        "DATALOADER.NUM_WORKERS", "0",
        "OUTPUT_DIR", str(branch["output"]),
    ]
    return cmd


for key in ["baseline", "local", "semantic", "combine"]:
    display(Markdown(f"### Evaluate: {BRANCHES[key]['title']}"))
    cmd = build_eval_command(key)
    run_command(cmd, cwd=BRANCHES[key]["source"], run=RUN_EVAL)

## 9. Bảng chỉ số tổng hợp và cách diễn giải

Cell kế tiếp đọc hai artifact:

- `metrics_summary.csv`: bảng tổng hợp các chỉ số truy hồi (mAP, Rank-1, Rank-5, Rank-10) giữa các nhánh.
- `latency_summary.csv`: bảng tổng hợp độ trễ suy luận phục vụ phân tích đánh đổi chính xác - hiệu năng.

Hai tệp này thường được tổng hợp từ log đánh giá. Nếu vừa chạy lại `test.py`, cần bảo đảm bước tổng hợp log sang CSV đã hoàn tất trước khi đọc bảng.

**Cách đọc nhanh các chỉ số:**

- **mAP** phản ánh chất lượng ranking toàn thể trên toàn bộ tập query.
- **Rank-1** phản ánh khả năng đưa đúng danh tính lên vị trí đầu tiên.
- **Rank-5/Rank-10** phản ánh độ ổn định khi xét nhiều vị trí đầu trong danh sách truy hồi.
- Trong bài toán Person ReID, cần đọc đồng thời mAP và CMC Rank-k để tránh diễn giải phiến diện chỉ dựa trên một metric đơn lẻ.

In [ ]:
# Cell này đọc các bảng tổng hợp metric/latency từ artifact đã chuẩn bị trước.
# Nếu tệp chưa tồn tại, show_csv sẽ in thông báo rõ để tránh nhầm lẫn khi thuyết trình.
show_csv(ARTIFACT_ROOT / "metrics_summary.csv")
show_csv(ARTIFACT_ROOT / "latency_summary.csv")

## 10. Phân tích định lượng qua trực quan hóa

Script `generate_report_visuals.py` tạo các biểu đồ phục vụ phân tích định lượng gồm:

- So sánh mAP và CMC Rank-k giữa các mô hình.
- Mức thay đổi so với baseline.
- Diễn biến đường học/đánh giá theo epoch (nếu có log tương ứng).
- Quan hệ đánh đổi giữa độ chính xác và độ trễ.

Các biểu đồ này giúp đối chiếu tính nhất quán giữa chỉ số bảng và xu hướng tổng thể của mô hình.

In [ ]:
# Cell này tạo/cập nhật các biểu đồ định lượng tổng hợp từ artifact đã có.
# Biểu đồ được hiển thị với kích thước lớn để thuận tiện cho phân tích khi trình bày video.
cmd = [PYTHON, str(TOOLS_DIR / "generate_report_visuals.py")]
run_command(cmd, cwd=ARTIFACT_ROOT, run=RUN_VISUALIZE)

quant_dir = ARTIFACT_ROOT / "visualizations" / "quantitative"
show_image(quant_dir / "09_summary_table.png", width=1000)
show_image(quant_dir / "01_metrics_grouped_bar.png", width=1000)
show_image(quant_dir / "02_delta_vs_baseline.png", width=900)
show_image(quant_dir / "06_validation_map_rank1_curves.png", width=1000)
show_image(quant_dir / "03_accuracy_latency_tradeoff.png", width=900)

## 11. Phân tích định tính thông qua Top-K retrieval

Top-K retrieval là một trong những trực quan hóa quan trọng nhất của Person ReID vì nó cho thấy trực tiếp cách mô hình xếp hạng toàn bộ gallery đối với một query.

**Nguyên tắc diễn giải:**

1. Query được so sánh với toàn bộ gallery bằng distance hoặc similarity trong không gian embedding.
2. Gallery được sắp theo thứ tự giảm dần của mức tương đồng hoặc tăng dần của khoảng cách.
3. Cần quan sát true match có xuất hiện ở các vị trí đầu hay không, và các false match có đặc điểm thị giác nào dẫn đến nhầm lẫn.
4. Khi true match không đứng đầu, cần phân tích xem lỗi đến từ pose mismatch, occlusion, background clutter hay visual ambiguity.

Phần này đặc biệt quan trọng vì phản ánh trực tiếp bản chất ranking của bài toán ReID.

In [ ]:
# Cell này tạo hình so sánh Top-K retrieval giữa 4 mô hình trên cùng tập truy vấn.
# Kết quả giúp đối chiếu trực tiếp thứ hạng true match giữa các nhánh phương pháp.
cmd = [
    PYTHON, str(TOOLS_DIR / "visualize_multimodel_retrieval.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(ARTIFACT_ROOT / "visualizations" / "qualitative"),
    "--topk", "5",
    "--num-queries", "3",
    "--num-distractors", "60",
    "--batch-size", "32",
]
run_command(cmd, cwd=ARTIFACT_ROOT, run=RUN_VISUALIZE)

qual_dir = ARTIFACT_ROOT / "visualizations" / "qualitative"
show_image(qual_dir / "01_same_query_topk_comparison.png", width=950)
show_image(qual_dir / "02_true_match_rank_matrix.png", width=1000)

## 12. Trực quan hóa Local Reliability

Mục tiêu của reliability heatmap là cho thấy mô hình đang phân bổ mức tin cậy không gian như thế nào trên ảnh người.

**Cách đọc reliability heatmap:**

- Vùng mang thông tin định danh như áo, quần, balo, giày hoặc dáng người thường nên có trọng số cao hơn.
- Background, occlusion, motion blur hoặc các vùng nhiễu thị giác thường nên có trọng số thấp hơn.
- Heatmap chỉ phản ánh mức độ đóng góp tương đối của từng vùng, vì vậy cần diễn giải cùng với kết quả retrieval chứ không tách rời.
- So sánh heatmap với Top-K retrieval giúp lý giải vì sao một truy hồi thành công hoặc thất bại.

Những minh họa này hỗ trợ diễn giải cơ chế hoạt động của nhánh Local Reliability ngoài các con số định lượng.

In [ ]:
# Cell này chạy bộ trực quan hóa chuyên biệt cho nhánh Local Reliability.
# Mục tiêu là quan sát đồng thời retrieval, heatmap độ tin cậy và các trường hợp thất bại.
local_out = ARTIFACT_ROOT / "visualizations" / "local_reliability"
cmd = [
    PYTHON, str(LOCAL_SRC / "visualize_person_reid_suite.py"),
    "--checkpoint", str(BRANCHES["local"]["checkpoint"]),
    "--config-file", str(BRANCHES["local"]["config"]),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(local_out),
    "--num-queries", "3",
    "--topk", "5",
    "--batch-size", "16",
]
run_command(cmd, cwd=LOCAL_SRC, run=RUN_VISUALIZE)

show_image(local_out / "01_ranked_topk_results.png", width=1000)
show_image(local_out / "03_multi_query_retrieval_grid.png", width=1000)
show_image(local_out / "05_failure_cases.png", width=850)
show_image(local_out / "07_reliability_heatmap.png", width=900)
show_image(local_out / "08_synthetic_occlusion_target.png", width=950)
show_image(local_out / "06_embedding_pca.png", width=800)

## 13. Trực quan hóa Semantic Alignment

Semantic Alignment nhằm đưa ràng buộc ngữ nghĩa theo vùng cơ thể vào biểu diễn patch token.

**Cách đọc semantic visualization:**

- Đối chiếu giữa mask hoặc body-part alignment và vị trí patch để đánh giá mức phù hợp hình học - ngữ nghĩa.
- Quan sát mức nhất quán semantic của cùng danh tính qua các camera khác nhau.
- Phân tích các vùng semantic dễ gây nhầm lẫn khi người có trang phục tương tự.
- Chất lượng semantic visualization phụ thuộc đáng kể vào chất lượng mask/parser; vì vậy cần tránh diễn giải quá mức là mô hình đã "hiểu" hoàn toàn cơ thể người.

Các hình này giúp kiểm tra giả thuyết rằng tín hiệu semantic cải thiện tính cấu trúc của embedding.

In [ ]:
# Cell này tạo trực quan hóa cho nhánh Semantic Alignment.
# Các hình đầu ra hỗ trợ phân tích mức phù hợp giữa semantic mask và biểu diễn patch token.
sem_out = ARTIFACT_ROOT / "visualizations" / "semantic"

cmd_basic = [
    PYTHON, str(SEMANTIC_SRC / "visualize_semantic_no_checkpoint.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
    "--num-samples", "3",
]
run_command(cmd_basic, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)

# Hai script dưới đây dùng human parser thật. Nếu máy chưa cache model, lần đầu có thể cần tải model.
cmd_real = [
    PYTHON, str(SEMANTIC_SRC / "visualize_real_semantic_examples.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
    "--num-images", "2",
]
cmd_extra = [
    PYTHON, str(SEMANTIC_SRC / "visualize_semantic_extra.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
]

run_command(cmd_real, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)
run_command(cmd_extra, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)

show_image(sem_out / "01_real_semantic_examples.png", width=1000)
show_image(sem_out / "02_real_vs_heuristic_mask.png", width=1000)
show_image(sem_out / "03_local_group_patch_targets.png", width=900)
show_image(sem_out / "04_semantic_patch_distribution.png", width=1000)
show_image(sem_out / "05_same_identity_semantic_consistency.png", width=980)
show_image(sem_out / "06_semantic_pipeline_diagram.png", width=1000)
show_image(sem_out / "07_semantic_overlay_grid.png", width=1000)

## 14. Phân tích failure case

Một báo cáo thực nghiệm đầy đủ cần trình bày cả trường hợp thành công và thất bại.

**Khung phân tích failure case đề xuất:**

1. Sai khác hình học lớn giữa query và gallery (pose/viewpoint mismatch).
2. Che khuất cục bộ làm mất thông tin nhận dạng quan trọng.
3. Nhiễu do trang phục tương tự hoặc nền phức tạp.
4. Mối liên hệ giữa vùng tin cậy thấp (reliability) và vị trí nhầm lẫn trong Top-K.

Failure case giúp xác định giới hạn hiện tại của phương pháp và cung cấp cơ sở cho các hướng cải tiến tiếp theo.

In [ ]:
# Cell này hiển thị nhanh nhóm hình liên quan failure case để phục vụ phân tích giới hạn mô hình.
# Nếu artifact chưa được sinh, notebook sẽ in thông báo rõ thay vì dừng đột ngột.
failure_candidates = [
    ARTIFACT_ROOT / "visualizations" / "local_reliability" / "05_failure_cases.png",
    ARTIFACT_ROOT / "visualizations" / "qualitative" / "01_same_query_topk_comparison.png",
    ARTIFACT_ROOT / "visualizations" / "qualitative" / "02_true_match_rank_matrix.png",
]

shown = 0
for img in failure_candidates:
    if img.exists():
        show_image(img, width=1100)
        shown += 1

if shown == 0:
    print("Chưa có artifact failure/qualitative. Hãy bật RUN_VISUALIZE = True để tạo lại hình.")

## 15. Phụ lục: lệnh huấn luyện và khả năng tái lập

Phần này cung cấp lệnh huấn luyện tương ứng cho từng nhánh nhằm phục vụ tái lập toàn bộ quy trình.

Trong bối cảnh quay video thuyết trình ngắn, không khuyến nghị chạy huấn luyện đầy đủ do chi phí thời gian lớn. Thay vào đó, nên sử dụng checkpoint đã huấn luyện sẵn để tập trung vào đánh giá và phân tích kết quả.

In [ ]:
# Cell này cung cấp lệnh huấn luyện tham khảo cho từng nhánh.
# Trong notebook demo chính thức, phần này thường chỉ được hiển thị chứ không chạy vì thời gian huấn luyện rất lớn.
def build_train_command(key, fast_demo=True):
    """Tạo command huấn luyện cho một nhánh mô hình.

    Parameters
    ----------
    key : str
        Tên nhánh trong `BRANCHES`.
    fast_demo : bool
        Nếu True, thêm override rút gọn để kiểm tra pipeline ở chế độ demo.

    Returns
    -------
    list
        Danh sách tham số lệnh huấn luyện cho subprocess.

    Why it exists
    ------------
    Notebook cần một lệnh train chuẩn hóa để người xem có thể tái lập pipeline,
    nhưng trong buổi thuyết trình chỉ nên dùng checkpoint đã có sẵn.
    """
    branch = BRANCHES[key]
    output_dir = DEMO_OUTPUT / f"train_{key}"
    cmd = [
        PYTHON, "train.py",
        "--config_file", branch["config"],
        "DATASETS.ROOT_DIR", str(DATA_PARENT),
        "MODEL.DEVICE_ID", DEVICE_ID,
        "OUTPUT_DIR", str(output_dir),
    ]
    if fast_demo:
        cmd += [
            "MODEL.PRETRAIN_CHOICE", "none",
            "SOLVER.MAX_EPOCHS", "1",
            "SOLVER.EVAL_PERIOD", "1",
            "SOLVER.CHECKPOINT_PERIOD", "1",
            "SOLVER.IMS_PER_BATCH", "8",
            "TEST.IMS_PER_BATCH", "64",
            "DATALOADER.NUM_WORKERS", "0",
        ]
    return cmd


for key in ["baseline", "local", "semantic", "combine"]:
    display(Markdown(f"### Train reference: {BRANCHES[key]['title']}"))
    cmd = build_train_command(key, fast_demo=FAST_TRAIN_DEMO)
    run_command(cmd, cwd=BRANCHES[key]["source"], run=RUN_TRAIN)

## 16. Kết luận thực nghiệm

Notebook đã hệ thống hóa quy trình đánh giá Person ReID từ dữ liệu, checkpoint, metric đến visualization. Kết quả định lượng cần được đọc thông qua mAP và CMC Rank-k, trong đó mAP phản ánh chất lượng ranking toàn cục còn Rank-k phản ánh khả năng đưa true match vào các vị trí đầu.

Các trực quan hóa Top-K retrieval cung cấp bằng chứng định tính về khả năng truy hồi đúng identity. Trong khi đó, reliability heatmap và semantic visualization hỗ trợ phân tích hành vi mô hình ở mức vùng ảnh, đặc biệt trong các trường hợp có occlusion, background clutter hoặc visual ambiguity.

Các failure case cho thấy bài toán vẫn còn thách thức khi người khác nhau có ngoại hình tương tự, khi ảnh bị che khuất, hoặc khi điều kiện camera làm thay đổi mạnh đặc trưng thị giác. Do đó, notebook không chỉ phục vụ demo, mà còn hỗ trợ phân tích và tái lập thực nghiệm.

## 17. Tóm tắt luận điểm trình bày

Gợi ý trình tự trình bày trong video theo phong cách học thuật:

1. Nêu phát biểu bài toán Person ReID và đặc trưng query-gallery ranking.
2. Trình bày baseline TransReID và động cơ của ba nhánh cải tiến.
3. Chứng minh điều kiện thực nghiệm hợp lệ thông qua preflight check.
4. Thực hiện đánh giá từ checkpoint để thu các chỉ số chuẩn.
5. Phân tích định lượng qua mAP/Rank-k và các biểu đồ tổng hợp.
6. Phân tích định tính qua Top-K retrieval, reliability heatmap và semantic visualization.
7. Thảo luận failure cases để làm rõ giới hạn của phương pháp.
8. Kết luận đóng góp, mức cải thiện quan sát được và hướng phát triển tiếp theo.